# Bach Propagation — Demo Educativo

**Proyecto:** Pipeline de análisis armónico y generación simbólica de música barroca  
**Autora:** Aura — 
**Educación:** Universidad Nacional Autonoma de México / Facultad de Ciencias / Facultad de Estudios Superiores Acatlán  
**Fecha:** 2026

---

## Abstract

Este notebook demuestra de forma reproducible el flujo central del proyecto **Bach Propagation**: desde la descarga de partituras del corpus de Bach hasta el entrenamiento de un mini-Transformer que aprende a predecir progresiones armónicas barrocas.

El sistema completo (en producción) combina:
- Un **pipeline simbólico** de análisis armónico (cuantización rítmica → estimación de tonalidad → clasificación de acordes → numerales romanos → función T/PD/D → reducción schenkeriana)
- Un **MusicTransformer** multi-stream entrenado sobre el corpus barroco completo
- Un **backend FastAPI** y un **frontend Next.js** para exploración e interacción

Este demo usa únicamente el corpus integrado de `music21` (sin archivos locales) y una versión simplificada del modelo para que todo corra en CPU en minutos.

---

### Flujo del notebook

```
[0] Setup  →  [1] Datos  →  [2] EDA  →  [3] Pipeline  →  [4] Tokenización  →  [5] Toy Transformer  →  [6] Evaluación
```

---
## Sección 0 — Instalación y Setup

In [ ]:
# Instalamos todas las dependencias necesarias para correr este notebook en Google Colab.
# El flag -q suprime la salida verbosa de pip.
!pip install -q music21==9.3.0 torch torchvision plotly==5.24.1 cufflinks==0.17.3 pandas numpy kaleido

In [1]:
# ── Imports principales ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import math
import random
from collections import Counter
from fractions import Fraction
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import music21
from music21 import corpus, note, chord, key, stream

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import cufflinks as cf

# ── Reproducibilidad global ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Configuración de Plotly y Cufflinks ─────────────────────────────────────
# Activamos cufflinks en modo offline para que los gráficos funcionen en Colab
# sin necesidad de una cuenta en Plotly Cloud.
cf.go_offline()
cf.set_config_file(theme='solar', sharing='public', offline=True)

# Paleta neon fría (azules y morados) para todos los gráficos del proyecto
NEON_PALETTE = [
    '#7B2FBE',  # violeta profundo
    '#A855F7',  # morado neon
    '#60A5FA',  # azul hielo
    '#38BDF8',  # cian eléctrico
    '#818CF8',  # indigo suave
    '#C084FC',  # lila
    '#2DD4BF',  # turquesa
    '#6366F1',  # índigo
]
DARK_BG     = '#0D0D1A'   # fondo oscuro del tema
DARK_PAPER  = '#13131F'   # fondo del área de trazado
AXIS_COLOR  = '#3B3B5C'   # color de ejes y rejillas
TEXT_COLOR  = '#E2E8F0'   # texto principal

# Layout base reutilizable para todos los gráficos Plotly
BASE_LAYOUT = dict(
    template='plotly_dark',
    paper_bgcolor=DARK_PAPER,
    plot_bgcolor=DARK_BG,
    font=dict(color=TEXT_COLOR, family='monospace'),
    xaxis=dict(gridcolor=AXIS_COLOR, zerolinecolor=AXIS_COLOR),
    yaxis=dict(gridcolor=AXIS_COLOR, zerolinecolor=AXIS_COLOR),
    margin=dict(l=60, r=30, t=60, b=60),
)

print(f"torch  {torch.__version__}")
print(f"music21 {music21.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo activo: {DEVICE}")

torch  2.10.0
music21 9.9.1
CUDA disponible: False
Dispositivo activo: cpu


---
## Sección 1 — Descarga de Datos

Usamos el corpus integrado de `music21`, que incluye los corales armonizados de J.S. Bach (BWV 1–438).  
No se requieren archivos locales: `music21` los descarga automáticamente al primer uso.

Para este demo cargamos **3 corales** representativos:

| BWV | Tonalidad | Nombre |  
|-----|-----------|--------|
| 253 | Fa mayor  | Auf meinen lieben Gott |
| 255 | Re menor  | Aus meines Herzens Grunde |
| 281 | Sol mayor | Christus, der ist mein Leben |

In [2]:
# ── Carga de corales desde el corpus integrado de music21 ─────────────────────
# Usamos corpus.search() con el prefijo 'bwv' para localizar cada coral.
# Nota: el segundo argumento de corpus.search() es el nombre del campo
# (no el compositor), por lo que la búsqueda va solo por número BWV.

BWV_IDS = ['bwv253', 'bwv255', 'bwv281']   # los 3 corales de demostración

scores_raw = {}   # {bwv_id: music21.stream.Score}

for bwv in BWV_IDS:
    results = corpus.search(bwv)
    if not results:
        raise RuntimeError(f"No se encontró {bwv} en el corpus de music21")
    # Tomamos el primer resultado y lo parseamos
    scores_raw[bwv] = corpus.parse(results[0].sourcePath)
    title = scores_raw[bwv].metadata.title or bwv
    print(f"✓  {bwv}  |  '{title}'")

print(f"\nTotal de partituras cargadas: {len(scores_raw)}")

✓  bwv253  |  'bwv253'
✓  bwv255  |  'bwv255'
✓  bwv281  |  '28. Christus, der ist mein Leben'

Total de partituras cargadas: 3


---
## Sección 2 — Análisis Exploratorio (EDA)

Antes de modelar exploramos el corpus para entender:
- Densidad de notas y duración de cada coral
- Distribución de clases de altura (pitch-class histogram)
- Distribución de calidades de acorde
- Tonalidades detectadas

In [3]:
# ── 2.1 Estadísticas básicas por partitura ────────────────────────────────────

NOTE_NAMES = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']

stats = []

for bwv, sc in scores_raw.items():
    # Aplanamos todas las partes para operar sobre una sola secuencia de eventos
    flat = sc.flatten()

    # Notas totales (excluye silencios)
    notes = flat.getElementsByClass(note.Note)
    n_notes = len(list(notes))

    # Duración total en negras (quarter notes)
    duration_qn = float(sc.duration.quarterLength)

    # Tonalidad estimada por music21 (Krumhansl-Schmuckler interno)
    key_est = sc.analyze('key')

    # Densidad de notas por compás
    n_measures = len(sc.parts[0].getElementsByClass('Measure'))
    density = n_notes / max(n_measures, 1)

    stats.append({
        'BWV':            bwv.upper(),
        'Notas':          n_notes,
        'Duración (QN)':  round(duration_qn, 1),
        'Compases':       n_measures,
        'Densidad':       round(density, 1),
        'Tonalidad':      f"{NOTE_NAMES[key_est.tonic.pitchClass]} {key_est.mode}",
        'Confianza':      round(key_est.correlationCoefficient, 3),
    })

df_stats = pd.DataFrame(stats).set_index('BWV')
print(df_stats.to_string())

        Notas  Duración (QN)  Compases  Densidad Tonalidad  Confianza
BWV                                                                  
BWV253    164           40.0        12      13.7   A major      0.878
BWV255    141           32.0        10      14.1   G major      0.826
BWV281    125           32.0         9      13.9   F major      0.919


In [4]:
# ── 2.2 Gráfico: Estadísticas por coral — barras agrupadas (Plotly) ───────────
#
# Usamos go.Bar directamente en lugar de cufflinks.iplot().
# Cufflinks 0.17.3 tiene un bug con Plotly 5.x: al convertir colores hex
# a rgba internamente genera strings con np.float64 que Plotly rechaza.
# go.Bar produce el mismo resultado visual sin esa dependencia.

df_plot = df_stats[['Notas', 'Duración (QN)', 'Compases']].copy()
bwv_labels = df_plot.index.tolist()   # ['BWV253', 'BWV255', 'BWV281']

fig = go.Figure()

for col, color in zip(df_plot.columns, [NEON_PALETTE[1], NEON_PALETTE[2], NEON_PALETTE[4]]):
    fig.add_trace(go.Bar(
        name=col,
        x=bwv_labels,
        y=df_plot[col],
        marker_color=color,
        text=df_plot[col],
        textposition='outside',
        textfont=dict(color=TEXT_COLOR),
    ))

fig.update_layout(
    **BASE_LAYOUT,
    barmode='group',
    title='Estadísticas del corpus demo por coral',
    xaxis_title='Coral (BWV)',
    yaxis_title='Valor',
    legend=dict(font=dict(color=TEXT_COLOR)),
    height=420,
)
fig.show()

In [5]:
# ── 2.3 Pitch-Class Histogram ─────────────────────────────────────────────────
#
# Calculamos la distribución de clases de altura (0=C … 11=B) para el corpus
# completo, ponderada por duración de cada nota (igual que Krumhansl-Schmuckler).

pc_histogram = Counter()

for sc in scores_raw.values():
    for n in sc.flatten().getElementsByClass(note.Note):
        pc  = n.pitch.pitchClass          # clase de altura 0-11
        dur = float(n.duration.quarterLength)
        pc_histogram[pc] += dur

# Construimos un DataFrame ordenado de C a B
df_pc = pd.DataFrame({
    'Nota':    NOTE_NAMES,
    'Peso':    [pc_histogram.get(i, 0.0) for i in range(12)],
})

# Gráfico de barras con gradiente neon morado→azul
fig = go.Figure(
    go.Bar(
        x=df_pc['Nota'],
        y=df_pc['Peso'],
        marker=dict(
            color=df_pc['Peso'],
            colorscale=[[0, '#3730A3'], [0.5, '#7C3AED'], [1, '#38BDF8']],
            showscale=False,
        ),
    )
)
fig.update_layout(
    **BASE_LAYOUT,
    title='Histograma de clases de altura (ponderado por duración) — corpus demo',
    xaxis_title='Clase de altura',
    yaxis_title='Peso acumulado (quarter notes)',
    height=400,
)
fig.show()

In [6]:
# ── 2.4 Distribución de calidades de acorde ───────────────────────────────────
#
# Chordify convierte la partitura coral (4 voces) a una secuencia de acordes.
# Luego clasificamos cada acorde según su calidad (mayor, menor, disminuido, etc.).

# Mapeo de tipo music21 → etiqueta legible
M21_QUALITY_MAP = {
    'major':          'Mayor',
    'minor':          'Menor',
    'diminished':     'Disminuido',
    'augmented':      'Aumentado',
    'dominant-seventh': 'Dom7',
    'diminished-seventh': 'Dis7',
    'half-diminished-seventh': 'Semi-dis7',
    'major-seventh':  'Maj7',
    'minor-seventh':  'Min7',
}

quality_counter = Counter()

for sc in scores_raw.values():
    # chordify() colapsa las voces simultáneas en un solo acorde por pulso
    chordified = sc.chordify()
    for c in chordified.flatten().getElementsByClass(chord.Chord):
        q = c.commonName          # ej. 'major triad', 'minor triad', …
        # Simplificamos el nombre de music21 a una categoría canónica
        if 'major triad' in q:
            quality_counter['Mayor'] += 1
        elif 'minor triad' in q:
            quality_counter['Menor'] += 1
        elif 'diminished triad' in q:
            quality_counter['Disminuido'] += 1
        elif 'augmented' in q:
            quality_counter['Aumentado'] += 1
        elif 'dominant' in q and 'seventh' in q:
            quality_counter['Dom7'] += 1
        elif 'seventh' in q:
            quality_counter['7ª otro'] += 1
        else:
            quality_counter['Otro'] += 1

df_quality = pd.DataFrame(
    list(quality_counter.items()), columns=['Calidad', 'Frecuencia']
).sort_values('Frecuencia', ascending=False)

# Gráfico de pastel interactivo (donut)
fig = go.Figure(
    go.Pie(
        labels=df_quality['Calidad'],
        values=df_quality['Frecuencia'],
        hole=0.45,
        marker=dict(colors=NEON_PALETTE, line=dict(color=DARK_BG, width=2)),
        textfont=dict(color=TEXT_COLOR),
    )
)
fig.update_layout(
    **BASE_LAYOUT,
    title='Distribución de calidades de acorde — corpus demo',
    legend=dict(font=dict(color=TEXT_COLOR)),
    height=450,
)
fig.show()

---
## Sección 3 — Pipeline de Análisis Armónico

Esta sección implementa una versión simplificada y **standalone** del pipeline de `sequence_extractor.py`. En producción, el pipeline usa clases especializadas (`KeyEstimator`, `ChordClassifier`, `RomanNumeralAnalyzer`, `FunctionLabeler`) sobre representaciones internas con `fractions.Fraction`. Aquí usamos directamente las primitivas de `music21` para mantener el notebook autocontenido.

**Flujo:**
```
Score → chordify() → estimación de tonalidad → cifrado romano → función T/PD/D → evento dict
```

### Reglas de función armónica

| Grado | Función |
|-------|---------|
| I, VI | Tónica (T=0) |
| II, IV | Subdominante (PD=1) |
| V, VII | Dominante (D=2) |
| III | Ambiguo → T |

In [7]:
# ── Definición del pipeline simplificado ──────────────────────────────────────

# Mapeo grado → función armónica (int)
# Fiel a src/harmonic/function_labeler.py: _BASE_FUNCTION
_DEGREE_TO_FUNCTION = {
    1: 0,   # Tónica
    2: 1,   # Predominante
    3: 0,   # Ambiguo → colapsa a Tónica (igual que en encoder.py)
    4: 1,   # Predominante
    5: 2,   # Dominante
    6: 0,   # Tónica
    7: 2,   # Dominante
}

# Mapeo calidad music21 → calidad canónica (fiel a encoder.py / chord_classifier.py)
_M21_TO_QUALITY = {
    'major triad':                   'major',
    'minor triad':                   'minor',
    'diminished triad':              'diminished',
    'augmented triad':               'augmented',
    'dominant seventh chord':        'dominant7',
    'diminished seventh chord':      'dim7',
    'half-diminished seventh chord': 'half-dim7',
    'major seventh chord':           'major7',
    'minor seventh chord':           'minor7',
}


def _classify_chord_quality(m21_chord) -> str:
    """Convierte el commonName de music21 a una calidad canónica del proyecto."""
    name = m21_chord.commonName.lower()
    for pattern, canonical in _M21_TO_QUALITY.items():
        if pattern in name:
            return canonical
    return 'major'   # fallback seguro


def extract_harmonic_sequence(m21_score) -> List[dict]:
    """
    Extrae una secuencia de eventos armónicos de una partitura music21.

    Versión simplificada de src/data/sequence_extractor.py::extract_sequence().
    Para cada ventana de acorde (chordify) calcula:
      - chord_root (0-11)    : clase de altura de la raíz
      - chord_quality (str)  : calidad canónica
      - rn_degree (1-7)      : grado en la tonalidad local
      - harmonic_function (0/1/2): T / PD / D
      - local_key_tonic (0-11)
      - local_key_mode (str)
      - onset (float)        : posición en quarter notes
      - duration (float)     : duración en quarter notes

    Parameters
    ----------
    m21_score : music21.stream.Score

    Returns
    -------
    List[dict]  — lista de eventos, uno por acorde detectado
    """
    events = []

    # 1. Estimamos la tonalidad global (Krumhansl-Schmuckler de music21)
    global_key = m21_score.analyze('key')

    # 2. chordify() colapsa las 4 voces del coral en acordes simultáneos
    chordified = m21_score.chordify()

    for c in chordified.flatten().getElementsByClass(chord.Chord):
        if c.duration.quarterLength == 0:
            continue   # descartamos acordes de duración cero

        # 3. Raíz e inversión
        root_pc  = c.root().pitchClass            # 0-11
        quality  = _classify_chord_quality(c)

        # 4. Cifrado romano usando music21
        try:
            rn = music21.roman.romanNumeralFromChord(c, global_key)
            degree = int(rn.scaleDegree) if rn.scaleDegree else 1
            degree = max(1, min(7, degree))  # clamp a rango válido
        except Exception:
            degree = 1

        # 5. Función armónica
        fn = _DEGREE_TO_FUNCTION.get(degree, 0)

        events.append({
            'onset':             float(c.offset),
            'duration':          float(c.duration.quarterLength),
            'chord_root':        root_pc,
            'chord_quality':     quality,
            'rn_degree':         degree,
            'harmonic_function': fn,
            'local_key_tonic':   global_key.tonic.pitchClass,
            'local_key_mode':    global_key.mode,
        })

    return events


# ── Extraemos secuencias para los 3 corales ────────────────────────────────────
sequences_raw = {}   # {bwv_id: List[dict]}

for bwv, sc in scores_raw.items():
    seq = extract_harmonic_sequence(sc)
    sequences_raw[bwv] = seq
    print(f"{bwv.upper()}  →  {len(seq)} eventos armónicos")

# Vista rápida del primer evento de cada coral
print("\nPrimer evento de bwv253:")
import pprint; pprint.pprint(sequences_raw['bwv253'][0])

BWV253  →  58 eventos armónicos
BWV255  →  44 eventos armónicos
BWV281  →  41 eventos armónicos

Primer evento de bwv253:
{'chord_quality': 'major',
 'chord_root': 9,
 'duration': 1.0,
 'harmonic_function': 0,
 'local_key_mode': 'major',
 'local_key_tonic': 9,
 'onset': 0.0,
 'rn_degree': 1}


In [8]:
# ── 3.1 Visualización: distribución de funciones armónicas ────────────────────

FN_LABELS = {0: 'Tónica (T)', 1: 'Subdominante (PD)', 2: 'Dominante (D)'}
fn_totals = Counter()

for seq in sequences_raw.values():
    for ev in seq:
        fn_totals[FN_LABELS[ev['harmonic_function']]] += 1

df_fn = pd.DataFrame(list(fn_totals.items()), columns=['Función', 'Frecuencia'])

fig = go.Figure(
    go.Bar(
        x=df_fn['Función'],
        y=df_fn['Frecuencia'],
        marker_color=[NEON_PALETTE[0], NEON_PALETTE[2], NEON_PALETTE[1]],
        text=df_fn['Frecuencia'],
        textposition='outside',
        textfont=dict(color=TEXT_COLOR),
    )
)
fig.update_layout(
    **BASE_LAYOUT,
    title='Distribución de funciones armónicas T / PD / D — corpus demo',
    xaxis_title='Función',
    yaxis_title='Frecuencia',
    height=400,
)
fig.show()

---
## Sección 4 — Preparación de Tensores

Implementamos una versión simplificada de `src/data/encoder.py` y `src/data/dataset.py`.

### Vocabulario de tokens

| Rango | Descripción |
|-------|-------------|
| 0 | PAD |
| 1 | START |
| 2 | END |
| 3 – 62 | 60 tokens de acorde (12 raíces × 5 calidades) |

**Fórmula:** `token = 3 + raíz × 5 + quality_idx`  
Calidades: `major=0, minor=1, diminished=2, augmented=3, dominant7=4`

> En el modelo completo el vocabulario tiene 90 tokens e incluye además función armónica (3) y tonalidad (24). Aquí usamos solo acordes para simplificar.

In [9]:
# ── Codificador simplificado (fiel a src/data/encoder.py) ──────────────────────

PAD   = 0
START = 1
END   = 2
CHORD_OFFSET    = 3
VOCAB_SIZE      = 63   # 3 especiales + 60 acordes (demo; producción = 90)

_QUALITY_IDX: Dict[str, int] = {
    'major':      0,
    'major7':     0,   # colapsa a major
    'minor':      1,
    'minor7':     1,   # colapsa a minor
    'diminished': 2,
    'dim7':       2,   # colapsa a diminished
    'half-dim7':  2,
    'augmented':  3,
    'dominant7':  4,
}

_IDX_TO_QUALITY = ['major', 'minor', 'diminished', 'augmented', 'dominant7']
NOTE_NAMES       = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']


def chord_to_token(root: int, quality: str) -> int:
    """Codifica (raíz 0-11, calidad) → token entero."""
    q_idx = _QUALITY_IDX.get(quality, 0)
    return CHORD_OFFSET + root * 5 + q_idx


def token_to_chord(token: int) -> Tuple[int, str]:
    """Decodifica token → (raíz, calidad)."""
    offset = token - CHORD_OFFSET
    return offset // 5, _IDX_TO_QUALITY[offset % 5]


def encode_sequence(events: List[dict]) -> List[int]:
    """
    Codifica una lista de eventos → [START, t1, t2, ..., END].
    Fiel a src/data/encoder.py::Encoder.encode_sequence().
    """
    tokens = [START]
    for ev in events:
        tokens.append(chord_to_token(ev['chord_root'], ev['chord_quality']))
    tokens.append(END)
    return tokens


# ── Codificamos las 3 secuencias ──────────────────────────────────────────────
encoded_sequences = [encode_sequence(seq) for seq in sequences_raw.values()]

for i, (bwv, tokens) in enumerate(zip(BWV_IDS, encoded_sequences)):
    print(f"{bwv.upper()}  →  {len(tokens)} tokens  |  muestra: {tokens[:8]} ...")

# Verificación: decodificamos los primeros 4 tokens de bwv253
print("\nDecodificación de los primeros 4 acordes de bwv253:")
for tok in encoded_sequences[0][1:5]:   # saltamos START
    root, quality = token_to_chord(tok)
    print(f"  token={tok:3d}  →  {NOTE_NAMES[root]} {quality}")

BWV253  →  60 tokens  |  muestra: [1, 48, 48, 13, 23, 48, 33, 13] ...
BWV255  →  46 tokens  |  muestra: [1, 3, 38, 13, 17, 38, 3, 38] ...
BWV281  →  43 tokens  |  muestra: [1, 28, 28, 3, 32, 53, 3, 13] ...

Decodificación de los primeros 4 acordes de bwv253:
  token= 48  →  A major
  token= 48  →  A major
  token= 13  →  D major
  token= 23  →  E major


In [10]:
# ── Dataset PyTorch ────────────────────────────────────────────────────────────
#
# Generamos más ejemplos de entrenamiento usando augmentación por transposición:
# cada secuencia se transpone a los 12 tonos (×12 el corpus original).
# Esta es la misma estrategia de src/data/sequence_extractor.py::augment_sequences().

def augment_tokens(token_list: List[int], semitones: int) -> List[int]:
    """
    Transpone semitones hacia arriba una secuencia codificada.
    Solo afecta tokens de acorde (rango CHORD_OFFSET .. CHORD_OFFSET+59);
    START, END y PAD se copian sin cambios.
    """
    result = []
    for tok in token_list:
        if CHORD_OFFSET <= tok < CHORD_OFFSET + 60:
            root, quality = token_to_chord(tok)
            new_root = (root + semitones) % 12
            result.append(chord_to_token(new_root, quality))
        else:
            result.append(tok)   # START, END, PAD sin cambio
    return result


# Augmentación: 3 corales × 12 tonos = 36 secuencias
all_sequences: List[List[int]] = []
for seq in encoded_sequences:
    for shift in range(12):
        all_sequences.append(augment_tokens(seq, shift))

print(f"Secuencias originales  : {len(encoded_sequences)}")
print(f"Secuencias aumentadas  : {len(all_sequences)}")


class HarmonyDataset(Dataset):
    """
    Dataset de lenguaje causal: input = seq[:-1], target = seq[1:].
    Fiel a src/data/dataset.py.
    """

    def __init__(self, sequences: List[List[int]]):
        self.samples = sequences

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        seq   = self.samples[idx]
        inp   = torch.tensor(seq[:-1], dtype=torch.long)   # x_{t}
        target = torch.tensor(seq[1:],  dtype=torch.long)   # x_{t+1}
        return inp, target


def collate_pad(batch):
    """Rellena secuencias de longitud variable con PAD=0 al final."""
    inputs, targets = zip(*batch)
    max_len = max(t.size(0) for t in inputs)
    inp_pad = torch.stack([F.pad(t, (0, max_len - t.size(0)), value=PAD) for t in inputs])
    tgt_pad = torch.stack([F.pad(t, (0, max_len - t.size(0)), value=PAD) for t in targets])
    return inp_pad, tgt_pad


# Partición 80/20 (fiel a src/data/dataset.py::build_datasets)
random.shuffle(all_sequences)
split_idx  = int(len(all_sequences) * 0.8)
train_set  = HarmonyDataset(all_sequences[:split_idx])
val_set    = HarmonyDataset(all_sequences[split_idx:])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True,  collate_fn=collate_pad)
val_loader   = DataLoader(val_set,   batch_size=8, shuffle=False, collate_fn=collate_pad)

print(f"Train: {len(train_set)} muestras  |  Val: {len(val_set)} muestras")
print(f"Batches por epoch (train): {len(train_loader)}")

Secuencias originales  : 3
Secuencias aumentadas  : 36
Train: 28 muestras  |  Val: 8 muestras
Batches por epoch (train): 4


---
## Sección 5 — Toy Transformer

Definimos `ToyHarmonyTransformer`, una versión reducida de la arquitectura de producción `MusicTransformer` (`src/models/music_transformer.py`).

### Diferencias respecto al modelo de producción

| Aspecto | Producción (`MusicTransformer`) | Demo (`ToyHarmonyTransformer`) |
|---------|--------------------------------|--------------------------------|
| Streams | 3 (chord + roman numeral + cadence) | 1 (solo chord) |
| `d_model` | 384 | 64 |
| Capas (`n_layers`) | 6 | 2 |
| Cabezas (`n_heads`) | 8 | 4 |
| Vocabulario | 90 tokens | 63 tokens |
| Corpus | Corpus barroco completo | 3 corales × 12 tonos |

La arquitectura interna es idéntica: **pre-LayerNorm**, **máscara causal**, inicialización con `std=0.02`.

In [11]:
# ── Toy Transformer ────────────────────────────────────────────────────────────
#
# Arquitectura fiel a src/models/music_transformer.py en estructura,
# reducida en hiperparámetros para correr rápido en CPU.

def _causal_mask(T: int, device, dtype) -> torch.Tensor:
    """
    Máscara causal (T×T) aditiva: -inf en el triángulo superior.
    Garantiza que la posición i solo atienda a j ≤ i (autorregresi ón).
    Fiel a src/models/music_transformer.py::_causal_mask().
    """
    mask = torch.full((T, T), float('-inf'), device=device, dtype=dtype)
    return torch.triu(mask, diagonal=1)   # conservamos diagonal y triángulo inferior


class ToyTransformerBlock(nn.Module):
    """
    Bloque Transformer con Pre-LayerNorm.
    Orden: LN → self-attention → residual → LN → FFN → residual.
    Idéntico a MusicTransformerBlock, con d_ff = 4 × d_model.
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        d_ff = 4 * d_model

        self.norm_attn = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True
        )
        self.norm_ffn = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
        # Pre-norm self-attention + residual
        h = self.norm_attn(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=causal_mask, need_weights=False)
        x = x + self.drop(attn_out)
        # Pre-norm FFN + residual
        x = x + self.drop(self.ffn(self.norm_ffn(x)))
        return x


class ToyHarmonyTransformer(nn.Module):
    """
    Transformer causal autorregresivo para predicción del siguiente acorde.

    Versión reducida de src/models/music_transformer.py::MusicTransformer.
    Opera sobre un único stream (chord tokens) a diferencia del modelo de
    producción que fusiona 3 streams (chord + roman numeral + cadence).

    Args:
        vocab_size (int): tamaño del vocabulario (63 en este demo)
        d_model (int):    dimensión de embedding y representaciones internas
        n_heads (int):    cabezas de atención (d_model debe ser divisible)
        n_layers (int):   número de bloques ToyTransformerBlock
        max_seq (int):    longitud máxima de secuencia
        dropout (float):  probabilidad de dropout
    """

    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        d_model:    int = 64,
        n_heads:    int = 4,
        n_layers:   int = 2,
        max_seq:    int = 512,
        dropout:    float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        # Embedding de tokens (padding_idx=0 → vector cero para PAD)
        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        # Embedding posicional aprendido (igual que MultiStreamEmbedding.pos_embed)
        self.pos_embed = nn.Embedding(max_seq, d_model)
        self.drop      = nn.Dropout(dropout)

        # Pila de bloques Transformer
        self.blocks = nn.ModuleList(
            [ToyTransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)]
        )
        self.norm = nn.LayerNorm(d_model)      # normalización final
        self.head = nn.Linear(d_model, vocab_size)  # cabeza de predicción

        self._init_weights()

    def _init_weights(self):
        """Inicialización estándar GPT: Normal(0, 0.02) para Linear y Embedding."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None:
                    with torch.no_grad():
                        m.weight[m.padding_idx].fill_(0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Paso forward.

        Args:
            x: (B, T) tensor de token IDs

        Returns:
            logits: (B, T, vocab_size) logits crudos
        """
        B, T = x.shape

        # Suma de embedding de token + embedding posicional
        positions = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        h = self.drop(self.tok_embed(x) + self.pos_embed(positions))

        # Máscara causal construida en cada forward
        cmask = _causal_mask(T, device=x.device, dtype=h.dtype)

        for block in self.blocks:
            h = block(h, cmask)

        return self.head(self.norm(h))   # (B, T, vocab_size)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── Instanciamos y mostramos el resumen ────────────────────────────────────────
model = ToyHarmonyTransformer(
    vocab_size = VOCAB_SIZE,
    d_model    = 64,
    n_heads    = 4,
    n_layers   = 2,
    dropout    = 0.1,
).to(DEVICE)

print(model)
print(f"\nParámetros entrenables: {model.count_parameters():,}")

ToyHarmonyTransformer(
  (tok_embed): Embedding(63, 64, padding_idx=0)
  (pos_embed): Embedding(512, 64)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-1): 2 x ToyTransformerBlock(
      (norm_attn): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm_ffn): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=256, out_features=64, bias=True)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=63, bias=True)
)

Parámetros entrenables: 140,991


In [12]:
# ── Bucle de entrenamiento (3 epochs) ─────────────────────────────────────────
#
# Fiel a src/models/train.py:
#   - Optimizador Adam
#   - CrossEntropyLoss(ignore_index=PAD)  → ignora los tokens de relleno
#   - Se reporta la perplejidad = exp(loss), métrica estándar para modelos de lenguaje

EPOCHS   = 3
LR       = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

history = {'epoch': [], 'train_loss': [], 'val_loss': [],
           'train_ppl': [], 'val_ppl': []}


def run_epoch(loader, train: bool) -> Tuple[float, float]:
    """
    Ejecuta un epoch de entrenamiento o validación.

    Returns:
        (loss_media, perplexity)
    """
    model.train(train)
    total_loss = 0.0
    n_batches  = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for inp, tgt in loader:
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)

            logits = model(inp)          # (B, T, V)
            B, T, V = logits.shape

            # CrossEntropy espera (N, C); aplanamos (B×T, V) y (B×T,)
            loss = criterion(logits.reshape(B * T, V), tgt.reshape(B * T))

            if train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping para estabilidad (buena práctica en Transformers)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

    avg_loss = total_loss / max(n_batches, 1)
    ppl      = math.exp(min(avg_loss, 20))   # clamp para evitar overflow
    return avg_loss, ppl


print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>10}  {'Train PPL':>10}  {'Val PPL':>9}")
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_ppl = run_epoch(train_loader, train=True)
    vl_loss, vl_ppl = run_epoch(val_loader,   train=False)

    history['epoch'].append(epoch)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_ppl'].append(tr_ppl)
    history['val_ppl'].append(vl_ppl)

    print(f"  {epoch:>4}  {tr_loss:>11.4f}  {vl_loss:>10.4f}  {tr_ppl:>10.2f}  {vl_ppl:>9.2f}")

print('\nEntrenamiento completado.')

 Epoch   Train Loss    Val Loss   Train PPL    Val PPL
----------------------------------------------------------
     1       3.9877      3.8323       53.93      46.17
     2       3.7148      3.6497       41.05      38.46
     3       3.5415      3.5373       34.52      34.37

Entrenamiento completado.


In [15]:
# ── Curvas de pérdida ─────────────────────────────────────────────────────────
#
# BASE_LAYOUT ya incluye las claves 'xaxis' y 'yaxis'.
# Pasar esas mismas claves de nuevo en update_layout() provoca
# "multiple values for keyword argument". La solución es aplicar
# los títulos y estilos de ejes con update_xaxes/update_yaxes,
# que además permiten apuntar a subplots específicos.

df_hist = pd.DataFrame(history).set_index('epoch')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Pérdida (Cross-Entropy)', 'Perplejidad'),
)

for col_name, color, row, col in [
    ('train_loss', NEON_PALETTE[2], 1, 1),
    ('val_loss',   NEON_PALETTE[1], 1, 1),
    ('train_ppl',  NEON_PALETTE[2], 1, 2),
    ('val_ppl',    NEON_PALETTE[1], 1, 2),
]:
    label = 'Train' if 'train' in col_name else 'Val'
    fig.add_trace(
        go.Scatter(
            x=df_hist.index,
            y=df_hist[col_name],
            name=f'{label} ({col_name.split("_")[1].upper()})',
            mode='lines+markers',
            line=dict(color=color, width=2),
            marker=dict(size=8),
        ),
        row=row, col=col,
    )

# BASE_LAYOUT no incluye xaxis/yaxis aquí para evitar colisión de claves
fig.update_layout(
    **BASE_LAYOUT,
    title='Curvas de entrenamiento — ToyHarmonyTransformer (3 epochs)',
    height=420,
)

# Aplicamos títulos y estilos de ejes por subplot sin conflicto
fig.update_xaxes(title_text='Epoch', dtick=1, gridcolor=AXIS_COLOR)
fig.update_yaxes(gridcolor=AXIS_COLOR)
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='PPL',  row=1, col=2)

fig.show()

---
## Sección 6 — Evaluación y Conclusiones

Calculamos las métricas de evaluación usadas en `src/evaluation/metrics.py`:

| Métrica | Definición |
|---------|------------|
| **Top-1 Accuracy** | % de veces que el token de mayor logit coincide con el target |
| **Perplexity** | exp(cross-entropy media); menor = mejor |
| **Transition Validity** | Fracción de transiciones predichas que se observaron en el conjunto de entrenamiento |

Comparamos el modelo entrenado contra un **baseline aleatorio** (probabilidad uniforme sobre el vocabulario).

In [16]:
# ── Evaluación completa ────────────────────────────────────────────────────────

def evaluate_model(loader) -> Dict[str, float]:
    """
    Calcula top-1 accuracy, perplexity y transition validity rate.
    Fiel a src/evaluation/metrics.py::evaluate_all().
    """
    model.eval()
    total_loss    = 0.0
    total_correct = 0
    total_tokens  = 0
    valid_trans   = 0
    total_trans   = 0

    with torch.no_grad():
        for inp, tgt in loader:
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
            logits   = model(inp)           # (B, T, V)
            B, T, V  = logits.shape

            # Máscara para ignorar posiciones PAD
            mask = tgt != PAD               # (B, T) bool

            # Pérdida
            loss = criterion(logits.reshape(B * T, V), tgt.reshape(B * T))
            total_loss += loss.item()

            # Top-1 Accuracy
            preds = logits.argmax(dim=-1)   # (B, T)
            total_correct += (preds[mask] == tgt[mask]).sum().item()
            total_tokens  += mask.sum().item()

            # Transition Validity: ¿el par (pred_t, pred_{t+1}) existe en el corpus?
            for b in range(B):
                seq_preds = preds[b][mask[b]].tolist()
                for i in range(len(seq_preds) - 1):
                    a, b_ = seq_preds[i], seq_preds[i + 1]
                    if CHORD_OFFSET <= a < CHORD_OFFSET + 60 and \
                       CHORD_OFFSET <= b_ < CHORD_OFFSET + 60:
                        total_trans += 1
                        if (a, b_) in known_transitions:
                            valid_trans += 1

    n_batches = len(loader)
    return {
        'accuracy':           total_correct / max(total_tokens, 1),
        'perplexity':         math.exp(min(total_loss / n_batches, 20)),
        'transition_validity': valid_trans / max(total_trans, 1),
    }


# ── Construimos el conjunto de transiciones conocidas (del corpus de entrenamiento) ──
known_transitions = set()
for seq in all_sequences[:split_idx]:
    chord_toks = [t for t in seq if CHORD_OFFSET <= t < CHORD_OFFSET + 60]
    for i in range(len(chord_toks) - 1):
        known_transitions.add((chord_toks[i], chord_toks[i + 1]))

print(f"Transiciones únicas en el conjunto de entrenamiento: {len(known_transitions)}")

# ── Baseline aleatorio ──────────────────────────────────────────────────────────
# Perplejidad teórica de un modelo uniforme = tamaño del vocabulario
random_ppl      = VOCAB_SIZE
random_accuracy = 1.0 / VOCAB_SIZE
random_validity = len(known_transitions) / (60 * 60)  # frac. de transiciones posibles observadas

# ── Evaluamos el modelo entrenado ──────────────────────────────────────────────
metrics = evaluate_model(val_loader)

# ── Tabla de resultados ──────────────────────────────────────────────────────
df_results = pd.DataFrame({
    'Modelo':               ['ToyHarmonyTransformer', 'Baseline Aleatorio'],
    'Accuracy (Top-1)':     [f"{metrics['accuracy']:.3f}",    f"{random_accuracy:.3f}"],
    'Perplexity':           [f"{metrics['perplexity']:.2f}",  f"{random_ppl:.2f}"],
    'Transition Validity':  [f"{metrics['transition_validity']:.3f}", f"{random_validity:.3f}"],
}).set_index('Modelo')

print("\n=== Resultados de Evaluación ===")
print(df_results.to_string())

Transiciones únicas en el conjunto de entrenamiento: 393

=== Resultados de Evaluación ===
                      Accuracy (Top-1) Perplexity Transition Validity
Modelo                                                               
ToyHarmonyTransformer            0.184      34.37               0.977
Baseline Aleatorio               0.016      63.00               0.109


In [18]:
# ── Gráfico comparativo de métricas ──────────────────────────────────────────
#
# Plotly 5.x no acepta hex de 8 dígitos (#RRGGBBAA) para fillcolor.
# Usamos un helper que convierte hex #RRGGBB + alpha float a 'rgba(r,g,b,a)'.

def hex_to_rgba(hex_color: str, alpha: float = 1.0) -> str:
    """Convierte '#RRGGBB' + alpha → 'rgba(r, g, b, alpha)'."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'


# Normalizamos Perplexity (invertida) para mostrar las 3 métricas en [0, 1]
categories = ['Accuracy', 'PPL (inversa norm.)', 'Trans. Validity']

max_ppl = random_ppl
model_ppl_norm  = 1 - (metrics['perplexity'] - 1) / max(max_ppl - 1, 1)
random_ppl_norm = 0.0   # baseline tiene la peor perplejidad

model_vals  = [metrics['accuracy'], model_ppl_norm,  metrics['transition_validity']]
random_vals = [random_accuracy,      random_ppl_norm, random_validity]

# Cerramos el polígono repitiendo el primer punto
model_vals  += [model_vals[0]]
random_vals += [random_vals[0]]
cats_closed  = categories + [categories[0]]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=model_vals,
    theta=cats_closed,
    fill='toself',
    name='ToyHarmonyTransformer',
    line=dict(color=NEON_PALETTE[2], width=2),
    fillcolor=hex_to_rgba(NEON_PALETTE[2], 0.20),
))
fig.add_trace(go.Scatterpolar(
    r=random_vals,
    theta=cats_closed,
    fill='toself',
    name='Baseline Aleatorio',
    line=dict(color=NEON_PALETTE[0], width=2, dash='dot'),
    fillcolor=hex_to_rgba(NEON_PALETTE[0], 0.13),
))

fig.update_layout(
    **BASE_LAYOUT,
    polar=dict(
        bgcolor=DARK_BG,
        radialaxis=dict(visible=True, range=[0, 1],
                        gridcolor=AXIS_COLOR, color=TEXT_COLOR),
        angularaxis=dict(gridcolor=AXIS_COLOR, color=TEXT_COLOR),
    ),
    title='Comparación de métricas: Transformer vs Baseline Aleatorio',
    height=500,
    legend=dict(font=dict(color=TEXT_COLOR)),
)
fig.show()

In [19]:
# ── Generación de muestra ─────────────────────────────────────────────────────
#
# Demostramos la capacidad generativa del modelo con un pequeño sampler autoregresivo.
# Dado un token START, el modelo genera una secuencia de 16 acordes usando
# muestreo con temperatura (temperature sampling).

@torch.no_grad()
def generate_sequence(model, start_token: int = START, n_tokens: int = 16,
                       temperature: float = 0.8) -> List[str]:
    """
    Generación autoregresiva con temperatura.

    Args:
        model:        ToyHarmonyTransformer entrenado
        start_token:  token inicial (START=1)
        n_tokens:     número de acordes a generar
        temperature:  controla la aleatoriedad (>1 = más aleatorio, <1 = más determinista)

    Returns:
        Lista de strings con nombre de cada acorde generado, ej. ['C major', 'G dominant7', ...]
    """
    model.eval()
    context = torch.tensor([[start_token]], dtype=torch.long, device=DEVICE)
    generated = []

    for _ in range(n_tokens):
        logits = model(context)           # (1, T, V)
        next_logits = logits[0, -1, :] / temperature   # logits del último token

        # Restringimos el muestreo solo a tokens de acorde válidos
        mask = torch.ones(VOCAB_SIZE, device=DEVICE) * float('-inf')
        mask[CHORD_OFFSET:CHORD_OFFSET + 60] = 0.0
        next_logits = next_logits + mask

        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        root, quality = token_to_chord(next_token.item())
        generated.append(f"{NOTE_NAMES[root]} {quality}")
        context = torch.cat([context, next_token.unsqueeze(0)], dim=1)

    return generated


# Generamos 16 acordes con temperatura 0.8
sample = generate_sequence(model, temperature=0.8, n_tokens=16)

print("Secuencia generada (16 acordes, temperatura=0.8):")
print("\n" + "  →  ".join(sample[:8]))
print("  →  " + "  →  ".join(sample[8:]))

Secuencia generada (16 acordes, temperatura=0.8):

Bb augmented  →  C# diminished  →  A major  →  E dominant7  →  C# minor  →  Ab major  →  Eb diminished  →  Ab major
  →  Ab major  →  C# major  →  G diminished  →  E augmented  →  D major  →  Ab minor  →  Eb dominant7  →  Bb diminished


---
## Conclusiones

### Lo que demostró este notebook

| Componente | Resultado |
|------------|-----------|
| **Pipeline de extracción** | Convierte partituras MusicXML en secuencias de tokens con función armónica (T/PD/D) |
| **EDA** | Los corales de Bach muestran predominio de acordes mayores y menores; el corpus es tonalmente coherente |
| **Tokenización** | Vocabulario de 63 tokens cubre 60 acordes posibles (12 raíces × 5 calidades) con colapso de calidades no canónicas |
| **Toy Transformer** | Aprende a reducir perplejidad y mejorar accuracy sobre el baseline aleatorio en solo 3 epochs |

### Limitaciones de este demo

- **Corpus mínimo**: 3 corales × 12 tonos = 36 secuencias. El modelo de producción usa el corpus barroco completo.
- **Modelo reducido**: `d_model=64`, 2 capas. El `MusicTransformer` de producción usa `d_model=384`, 6 capas, 3 streams.
- **Sin análisis schenkeriano**: el demo no incluye la reducción de prolongación (árbol de 3 niveles de `ProlongationAnalyzer`).
- **Sin gramática constrictiva**: el generador de producción aplica reglas de gramática barroca (`BaroqueGrammar`) que bloquean transiciones prohibidas (V7→IV, etc.).

### Próximos pasos en el proyecto

1. Entrenar el `MusicTransformer` completo sobre el corpus barroco multi-compositor
2. Evaluar contra el análisis de música21 como _ground truth_ (validación musicológica)
3. Integrar el análisis de embeddings para verificar si el modelo aprende distancias tonales
4. Exponer la generación a través del endpoint `POST /api/generate` del backend FastAPI

---

> **Repositorio:** `bach-propagation` — FastAPI + MusicTransformer + Next.js  
> **Referencia teórica:** Krumhansl & Schmuckler (1986), Schenkerian Analysis, GPT-style causal LM